In [ ]:
# ==============================
# RAG CHATBOT USING LANGCHAIN
# ==============================

# Import required libraries
import os
from datasets import load_dataset

# Text splitting for long documents
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Vector database
from langchain_community.vectorstores import FAISS

# Embedding model
from langchain_community.embeddings import HuggingFaceEmbeddings

# RAG chain
from langchain.chains import RetrievalQA

# Groq LLM
from langchain_groq import ChatGroq

In [ ]:
# Load Groq API key from Google Colab secrets
from google.colab import userdata
import os

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

In [ ]:
# Load SQuAD dataset from HuggingFace
dataset = load_dataset("squad", split="train[:2000]")

# Extract context text
documents = [item["context"] for item in dataset]

print("Loaded documents:", len(documents))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loaded documents: 2000


In [ ]:
# Split large text into smaller chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

docs = text_splitter.create_documents(documents)

print("Chunks created:", len(docs))

Chunks created: 7395


In [ ]:
# Convert text chunks into vector embeddings
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/tmp/ipykernel_24501/278841696.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFaceEmbeddings`.
  embeddings = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# Store embeddings inside FAISS vector database
vectorstore = FAISS.from_documents(
    docs,
    embeddings
)

print("Vector database created")

Vector database created


In [ ]:
# Retriever searches the vector database
# k=3 means retrieve top 3 relevant chunks
retriever = vectorstore.as_retriever(
    search_kwargs={"k":3}
)

In [ ]:
# Load Groq Llama model
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0
)

In [ ]:
# Combine retriever + LLM
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever
)

In [ ]:
# Simple chatbot interface
while True:

    query = input("Ask a question: ")

    # Exit condition
    if query.lower() == "exit":
        break

    # Generate answer
    answer = qa_chain.run(query)

    print("\nAnswer:", answer)

Ask a question: What is the capital of the france?


/tmp/ipykernel_24501/832971115.py:11: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use invoke instead.
  answer = qa_chain.run(query)



Answer: The capital of France is Paris.
